# UPI Shield AI — Notebook 3: Feature Engineering

## Objective

The objective of this notebook is to transform the raw transaction dataset into a clean, meaningful, machine-learning-ready dataset.

In Notebook 2, we found that fraud patterns may be related to transaction type, transaction amount, sender balance behavior, receiver balance behavior, and balance mismatch.

In this notebook, we will create new features such as:

1. `balanceDiffOrig`
2. `balanceDiffDest`
3. `errorBalanceOrig`
4. `errorBalanceDest`
5. `isZeroBalanceAfterTransaction`
6. `isHighAmount`
7. `hourOfDay`

Important Note:

SMOTE will not be applied in this notebook. SMOTE must be applied only after train-test split in the model training notebook to avoid data leakage.

In [19]:
# ---------------------------------------------------------
# Step 1: Import Required Libraries
# ---------------------------------------------------------
# We import libraries required for data loading, feature
# engineering, file handling, and saving processed outputs.

import os
import json
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# Display settings for better readability
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

print("Libraries imported successfully.")

Libraries imported successfully.


In [20]:
# ---------------------------------------------------------
# Step 2: Mount Google Drive
# ---------------------------------------------------------
# We mount Google Drive to access the project dataset and
# save processed files permanently.

from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.


In [21]:
# ---------------------------------------------------------
# Step 3: Define Project Paths
# ---------------------------------------------------------
# BASE_DIR is the main project directory.
# RAW_DATA_DIR contains the original dataset.
# PROCESSED_DATA_DIR stores cleaned and processed datasets.
# REPORTS_DIR stores text reports and summaries.

BASE_DIR = "/content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring"

RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, "data", "processed")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")

DATA_PATH = os.path.join(RAW_DATA_DIR, "onlinefraud.csv")

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("Base Directory:", BASE_DIR)
print("Raw Data Directory:", RAW_DATA_DIR)
print("Processed Data Directory:", PROCESSED_DATA_DIR)
print("Reports Directory:", REPORTS_DIR)
print("Dataset Path:", DATA_PATH)

Base Directory: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring
Raw Data Directory: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/data/raw
Processed Data Directory: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/data/processed
Reports Directory: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/reports
Dataset Path: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/data/raw/onlinefraud.csv


In [22]:
# ---------------------------------------------------------
# Step 4: Load Only Required Columns
# ---------------------------------------------------------
# The original dataset is very large.
# To reduce RAM usage, we load only the columns required
# for feature engineering and model training.

required_columns = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFraud"
]

df = pd.read_csv(DATA_PATH, usecols=required_columns)

print("Dataset loaded successfully with required columns only.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully with required columns only.
Shape: (6362620, 8)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.6400,170136.0000,160296.3600,0.0000,0.0000,0
1,1,PAYMENT,1864.2800,21249.0000,19384.7200,0.0000,0.0000,0
2,1,TRANSFER,181.0000,181.0000,0.0000,0.0000,0.0000,1
3,1,CASH_OUT,181.0000,181.0000,0.0000,21182.0000,0.0000,1
4,1,PAYMENT,11668.1400,41554.0000,29885.8600,0.0000,0.0000,0


In [23]:
# ---------------------------------------------------------
# Step 5: Optimize Data Types
# ---------------------------------------------------------
# We reduce memory usage by converting columns to smaller
# data types. This is important in Google Colab.

df["step"] = df["step"].astype("int16")
df["type"] = df["type"].astype("category")
df["amount"] = df["amount"].astype("float32")
df["oldbalanceOrg"] = df["oldbalanceOrg"].astype("float32")
df["newbalanceOrig"] = df["newbalanceOrig"].astype("float32")
df["oldbalanceDest"] = df["oldbalanceDest"].astype("float32")
df["newbalanceDest"] = df["newbalanceDest"].astype("float32")
df["isFraud"] = df["isFraud"].astype("int8")

memory_usage = df.memory_usage(deep=True).sum() / (1024 ** 2)

print("Data types optimized successfully.")
print("Memory usage after optimization: {:.2f} MB".format(memory_usage))

df.dtypes

Data types optimized successfully.
Memory usage after optimization: 145.63 MB


,0
step,int16
type,category
amount,float32
oldbalanceOrg,float32
newbalanceOrig,float32
oldbalanceDest,float32
newbalanceDest,float32
isFraud,int8


In [24]:
# ---------------------------------------------------------
# Step 6: Basic Dataset Check
# ---------------------------------------------------------
# Before sampling and feature engineering, we quickly check
# missing values and target distribution.

print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["isFraud"].value_counts())

print("\nTarget percentage:")
print(df["isFraud"].value_counts(normalize=True) * 100)

Dataset shape: (6362620, 8)

Missing values:
step              0
type              0
amount            0
oldbalanceOrg     0
newbalanceOrig    0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
dtype: int64

Target distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Target percentage:
isFraud
0   99.8709
1    0.1291
Name: proportion, dtype: float64


In [25]:
# ---------------------------------------------------------
# Step 7: Create RAM-Safe Modeling Dataset
# ---------------------------------------------------------
# Full dataset is too large for repeated processing in Colab.
#
# Strategy:
# 1. Keep all fraud transactions.
# 2. Randomly sample normal transactions.
# 3. Combine them and shuffle.
#
# This keeps all fraud cases and reduces RAM usage.

fraud_df = df[df["isFraud"] == 1]
normal_df = df[df["isFraud"] == 0]

# Start with 200000.
# If RAM still crashes, change this to 100000.
NORMAL_SAMPLE_SIZE = 200000

normal_sample = normal_df.sample(
    n=NORMAL_SAMPLE_SIZE,
    random_state=42
)

modeling_df = pd.concat([fraud_df, normal_sample], axis=0)

modeling_df = modeling_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Fraud records kept:", fraud_df.shape[0])
print("Normal records sampled:", normal_sample.shape[0])
print("Final modeling dataset shape:", modeling_df.shape)

print("\nFinal modeling dataset target distribution:")
print(modeling_df["isFraud"].value_counts())

print("\nFinal modeling dataset target percentage:")
print(modeling_df["isFraud"].value_counts(normalize=True) * 100)

modeling_df.head()

Fraud records kept: 8213
Normal records sampled: 200000
Final modeling dataset shape: (208213, 8)

Final modeling dataset target distribution:
isFraud
0    200000
1      8213
Name: count, dtype: int64

Final modeling dataset target percentage:
isFraud
0   96.0555
1    3.9445
Name: proportion, dtype: float64


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,349,PAYMENT,65487.9883,141478.0000,75990.0078,0.0000,0.0000,0
1,257,CASH_OUT,299940.0312,0.0000,0.0000,502460.1250,802400.1250,0
2,281,PAYMENT,16878.8594,17579.6504,700.7900,0.0000,0.0000,0
3,42,CASH_OUT,132800.6250,0.0000,0.0000,638717.3125,771517.9375,0
4,15,CASH_OUT,238788.1562,0.0000,0.0000,672467.0625,911255.2500,0


In [26]:
# ---------------------------------------------------------
# Step 8: Free RAM
# ---------------------------------------------------------
# After creating the smaller modeling dataset, we delete
# large temporary dataframes to prevent Colab RAM crash.

del df
del normal_df
del fraud_df
del normal_sample

import gc
gc.collect()

print("Unused large dataframes deleted from memory.")

Unused large dataframes deleted from memory.


In [27]:
# ---------------------------------------------------------
# Step 9: Create Feature Engineering Copy
# ---------------------------------------------------------
# We perform feature engineering on the RAM-safe modeling dataset,
# not the full dataset.

feature_df = modeling_df.copy()

print("Feature engineering dataset created.")
print("Shape:", feature_df.shape)

feature_df.head()

Feature engineering dataset created.
Shape: (208213, 8)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,349,PAYMENT,65487.9883,141478.0000,75990.0078,0.0000,0.0000,0
1,257,CASH_OUT,299940.0312,0.0000,0.0000,502460.1250,802400.1250,0
2,281,PAYMENT,16878.8594,17579.6504,700.7900,0.0000,0.0000,0
3,42,CASH_OUT,132800.6250,0.0000,0.0000,638717.3125,771517.9375,0
4,15,CASH_OUT,238788.1562,0.0000,0.0000,672467.0625,911255.2500,0


## Feature Engineering Plan

We will create transaction-behavior features.

### 1. Sender Balance Difference

`balanceDiffOrig = oldbalanceOrg - newbalanceOrig`

This shows how much money was deducted from the sender account.

### 2. Receiver Balance Difference

`balanceDiffDest = newbalanceDest - oldbalanceDest`

This shows how much money was added to the receiver account.

### 3. Sender Balance Error

`errorBalanceOrig = oldbalanceOrg - amount - newbalanceOrig`

This checks whether sender balance movement matches the transaction amount.

### 4. Receiver Balance Error

`errorBalanceDest = oldbalanceDest + amount - newbalanceDest`

This checks whether receiver balance movement matches the transaction amount.

### 5. Zero Balance Indicator

`isZeroBalanceAfterTransaction = 1 if newbalanceOrig == 0 else 0`

This checks whether sender balance becomes zero after transaction.

### 6. High Amount Indicator

`isHighAmount = 1 if amount is greater than 95th percentile else 0`

This identifies unusually large transactions.

### 7. Hour of Day

`hourOfDay = step % 24`

Since one step represents one hour, this gives an approximate hour of transaction.

In [28]:
# ---------------------------------------------------------
# Step 7: Create Balance Difference Features
# ---------------------------------------------------------
# balanceDiffOrig shows how much money was deducted from the sender.
# balanceDiffDest shows how much money was added to the receiver.

feature_df["balanceDiffOrig"] = feature_df["oldbalanceOrg"] - feature_df["newbalanceOrig"]
feature_df["balanceDiffDest"] = feature_df["newbalanceDest"] - feature_df["oldbalanceDest"]

feature_df[[
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "balanceDiffOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "balanceDiffDest",
    "isFraud"
]].head()

,amount,oldbalanceOrg,newbalanceOrig,balanceDiffOrig,oldbalanceDest,newbalanceDest,balanceDiffDest,isFraud
0,65487.9883,141478.0000,75990.0078,65487.9922,0.0000,0.0000,0.0000,0
1,299940.0312,0.0000,0.0000,0.0000,502460.1250,802400.1250,299940.0000,0
2,16878.8594,17579.6504,700.7900,16878.8613,0.0000,0.0000,0.0000,0
3,132800.6250,0.0000,0.0000,0.0000,638717.3125,771517.9375,132800.6250,0
4,238788.1562,0.0000,0.0000,0.0000,672467.0625,911255.2500,238788.1875,0


In [29]:
# ---------------------------------------------------------
# Step 8: Create Balance Error Features
# ---------------------------------------------------------
# These features compare expected balance with actual balance.
#
# Sender expected balance:
# oldbalanceOrg - amount = newbalanceOrig
#
# Receiver expected balance:
# oldbalanceDest + amount = newbalanceDest
#
# Any mismatch is captured as balance error.

feature_df["errorBalanceOrig"] = (
    feature_df["oldbalanceOrg"] - feature_df["amount"] - feature_df["newbalanceOrig"]
)

feature_df["errorBalanceDest"] = (
    feature_df["oldbalanceDest"] + feature_df["amount"] - feature_df["newbalanceDest"]
)

feature_df[[
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "errorBalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "errorBalanceDest",
    "isFraud"
]].head()

,amount,oldbalanceOrg,newbalanceOrig,errorBalanceOrig,oldbalanceDest,newbalanceDest,errorBalanceDest,isFraud
0,65487.9883,141478.0000,75990.0078,0.0078,0.0000,0.0000,65487.9883,0
1,299940.0312,0.0000,0.0000,-299940.0312,502460.1250,802400.1250,0.0000,0
2,16878.8594,17579.6504,700.7900,0.0010,0.0000,0.0000,16878.8594,0
3,132800.6250,0.0000,0.0000,-132800.6250,638717.3125,771517.9375,0.0000,0
4,238788.1562,0.0000,0.0000,-238788.1562,672467.0625,911255.2500,0.0000,0


In [30]:
# ---------------------------------------------------------
# Step 9: Create Zero Balance Indicator
# ---------------------------------------------------------
# This feature checks whether the sender's balance becomes
# zero after the transaction.
#
# 1 = Sender balance became zero
# 0 = Sender balance did not become zero

feature_df["isZeroBalanceAfterTransaction"] = np.where(
    feature_df["newbalanceOrig"] == 0,
    1,
    0
)

feature_df[[
    "oldbalanceOrg",
    "newbalanceOrig",
    "isZeroBalanceAfterTransaction",
    "isFraud"
]].head()

,oldbalanceOrg,newbalanceOrig,isZeroBalanceAfterTransaction,isFraud
0,141478.0000,75990.0078,0,0
1,0.0000,0.0000,1,0
2,17579.6504,700.7900,0,0
3,0.0000,0.0000,1,0
4,0.0000,0.0000,1,0


In [31]:
# ---------------------------------------------------------
# Step 10: Create High Amount Indicator
# ---------------------------------------------------------
# We mark transactions as high amount if the transaction amount
# is greater than or equal to the 95th percentile.
#
# This threshold is saved because the Streamlit app will need
# the same threshold for future predictions.

HIGH_AMOUNT_THRESHOLD = feature_df["amount"].quantile(0.95)

feature_df["isHighAmount"] = np.where(
    feature_df["amount"] >= HIGH_AMOUNT_THRESHOLD,
    1,
    0
)

print("High amount threshold:", HIGH_AMOUNT_THRESHOLD)

feature_df[["amount", "isHighAmount", "isFraud"]].head()

High amount threshold: 652597.4249999999


,amount,isHighAmount,isFraud
0,65487.9883,0,0
1,299940.0312,0,0
2,16878.8594,0,0
3,132800.6250,0,0
4,238788.1562,0,0


In [32]:
# ---------------------------------------------------------
# Step 11: Create Hour of Day Feature
# ---------------------------------------------------------
# In PaySim, the step column represents time in hours.
# We create hourOfDay using modulo 24.
#
# Example:
# step 25 means hour 1 of the next day.

feature_df["hourOfDay"] = feature_df["step"] % 24

feature_df[["step", "hourOfDay", "isFraud"]].head()

,step,hourOfDay,isFraud
0,349,13,0
1,257,17,0
2,281,17,0
3,42,18,0
4,15,15,0


In [33]:
# ---------------------------------------------------------
# Step 12: Create Transaction Type Risk Feature
# ---------------------------------------------------------
# From EDA, fraud usually appears in specific transaction types.
# We create a simple rule-based feature:
#
# 1 = Higher-risk transaction type
# 0 = Lower-risk transaction type
#
# In PaySim, fraud is mainly present in TRANSFER and CASH_OUT.

high_risk_types = ["TRANSFER", "CASH_OUT"]

feature_df["isHighRiskType"] = np.where(
    feature_df["type"].isin(high_risk_types),
    1,
    0
)

feature_df[["type", "isHighRiskType", "isFraud"]].head()

,type,isHighRiskType,isFraud
0,PAYMENT,0,0
1,CASH_OUT,1,0
2,PAYMENT,0,0
3,CASH_OUT,1,0
4,CASH_OUT,1,0


In [34]:
# ---------------------------------------------------------
# Step 13: Create Log Amount Feature
# ---------------------------------------------------------
# Transaction amount can be highly skewed.
# log1p reduces skewness and safely handles zero values.
#
# log1p(x) = log(1 + x)

feature_df["logAmount"] = np.log1p(feature_df["amount"])

feature_df[["amount", "logAmount", "isFraud"]].head()

,amount,logAmount,isFraud
0,65487.9883,11.0896,0
1,299940.0312,12.6113,0
2,16878.8594,9.7339,0
3,132800.6250,11.7966,0
4,238788.1562,12.3833,0


In [35]:
# ---------------------------------------------------------
# Step 14: Create Log Balance-Based Features
# ---------------------------------------------------------
# Balance difference and balance error values can be large.
# We create log-transformed absolute versions to reduce skewness.
#
# abs() is used because balance errors can be negative.

feature_df["logBalanceDiffOrig"] = np.log1p(np.abs(feature_df["balanceDiffOrig"]))
feature_df["logBalanceDiffDest"] = np.log1p(np.abs(feature_df["balanceDiffDest"]))
feature_df["logErrorBalanceOrig"] = np.log1p(np.abs(feature_df["errorBalanceOrig"]))
feature_df["logErrorBalanceDest"] = np.log1p(np.abs(feature_df["errorBalanceDest"]))

feature_df[[
    "balanceDiffOrig",
    "logBalanceDiffOrig",
    "balanceDiffDest",
    "logBalanceDiffDest",
    "errorBalanceOrig",
    "logErrorBalanceOrig",
    "errorBalanceDest",
    "logErrorBalanceDest"
]].head()

,balanceDiffOrig,logBalanceDiffOrig,balanceDiffDest,logBalanceDiffDest,errorBalanceOrig,logErrorBalanceOrig,errorBalanceDest,logErrorBalanceDest
0,65487.9922,11.0896,0.0000,0.0000,0.0078,0.0078,65487.9883,11.0896
1,0.0000,0.0000,299940.0000,12.6113,-299940.0312,12.6113,0.0000,0.0000
2,16878.8613,9.7339,0.0000,0.0000,0.0010,0.0010,16878.8594,9.7339
3,0.0000,0.0000,132800.6250,11.7966,-132800.6250,11.7966,0.0000,0.0000
4,0.0000,0.0000,238788.1875,12.3833,-238788.1562,12.3833,0.0000,0.0000


In [36]:
# ---------------------------------------------------------
# Step 15: Check New Columns
# ---------------------------------------------------------
# This cell displays all columns after feature engineering.

print("Total columns after feature engineering:", feature_df.shape[1])

print("\nColumns:")
for col in feature_df.columns:
    print("-", col)

Total columns after feature engineering: 21

Columns:
- step
- type
- amount
- oldbalanceOrg
- newbalanceOrig
- oldbalanceDest
- newbalanceDest
- isFraud
- balanceDiffOrig
- balanceDiffDest
- errorBalanceOrig
- errorBalanceDest
- isZeroBalanceAfterTransaction
- isHighAmount
- hourOfDay
- isHighRiskType
- logAmount
- logBalanceDiffOrig
- logBalanceDiffDest
- logErrorBalanceOrig
- logErrorBalanceDest


In [37]:
# ---------------------------------------------------------
# Step 16: Engineered Feature Summary
# ---------------------------------------------------------
# This gives a statistical summary of newly created features.

engineered_features = [
    "balanceDiffOrig",
    "balanceDiffDest",
    "errorBalanceOrig",
    "errorBalanceDest",
    "isZeroBalanceAfterTransaction",
    "isHighAmount",
    "hourOfDay",
    "isHighRiskType",
    "logAmount",
    "logBalanceDiffOrig",
    "logBalanceDiffDest",
    "logErrorBalanceOrig",
    "logErrorBalanceDest"
]

feature_df[engineered_features].describe()

,balanceDiffOrig,balanceDiffDest,errorBalanceOrig,errorBalanceDest,isZeroBalanceAfterTransaction,isHighAmount,hourOfDay,isHighRiskType,logAmount,logBalanceDiffOrig,logBalanceDiffDest,logErrorBalanceOrig,logErrorBalanceDest
count,208213.0000,208213.0000,208213.0000,208213.0000,208213.0000,208213.0000,208213.0000,208213.0000,208213.0000,208213.0000,208213.0000,208213.0000,208213.0000
mean,35304.1875,149360.8906,-195719.3438,81662.6406,0.5855,0.0500,15.1762,0.4575,10.9221,6.9209,7.4618,8.6026,6.2674
std,565744.5000,868278.1250,614414.2500,550970.1250,0.4926,0.2179,4.5077,0.4982,1.8629,5.0256,5.8154,5.1110,5.3553
min,-1782621.0000,-4525019.0000,-69886728.0000,-56308420.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,0.0000,0.0000,-240479.3438,0.0000,0.0000,0.0000,12.0000,0.0000,9.5501,0.0000,0.0000,6.3816,0.0000
50%,0.0000,0.0000,-57193.1797,3479.0300,1.0000,0.0000,16.0000,0.0000,11.3016,9.0666,11.0225,10.9542,8.5466
75%,11624.2422,154061.2500,-589.8900,32139.6895,1.0000,0.0000,19.0000,1.0000,12.3034,11.0669,12.2443,12.3904,10.8084
max,10000002.0000,82704592.0000,2.0000,10000000.0000,1.0000,1.0000,23.0000,1.0000,18.0624,16.1181,18.2308,18.0624,17.8464


In [38]:
# ---------------------------------------------------------
# Step 17: Engineered Features by Fraud Class
# ---------------------------------------------------------
# This helps us see how engineered features behave differently
# for normal and fraud transactions.

feature_class_summary = feature_df.groupby("isFraud")[engineered_features].mean()

feature_class_summary.index = ["Normal Transactions", "Fraud Transactions"]

feature_class_summary

,balanceDiffOrig,balanceDiffDest,errorBalanceOrig,errorBalanceDest,isZeroBalanceAfterTransaction,isHighAmount,hourOfDay,isHighRiskType,logAmount,logBalanceDiffOrig,logBalanceDiffDest,logErrorBalanceOrig,logErrorBalanceDest
Normal Transactions,-23089.0371,125292.8125,-203317.4688,54935.6289,0.5692,0.0349,15.3252,0.4353,10.8412,6.6785,7.5016,8.9528,6.2511
Fraud Transactions,1457274.8750,735458.0000,-10692.3262,732509.3125,0.9805,0.4173,11.5465,1.0000,12.8920,12.8244,6.4930,0.0761,6.6637


## Feature Selection Plan

For model training, we will remove columns that are not directly suitable.

### Columns to remove

1. `nameOrig`
2. `nameDest`

These are sender and receiver IDs. They have high cardinality and may cause the model to memorize accounts instead of learning transaction behavior.

### Column to handle carefully

`isFlaggedFraud`

This is a rule-based fraud flag already present in the dataset. We will remove it from training to avoid giving the model a shortcut.

### Categorical column

`type`

This will be converted into numeric format using one-hot encoding.

In [39]:
# ---------------------------------------------------------
# Step 18: Prepare Dataset for Encoding
# ---------------------------------------------------------
# In the RAM-safe version, we already avoided loading:
# nameOrig, nameDest, and isFlaggedFraud.
#
# Therefore, no ID/leakage columns need to be dropped here.

model_df = feature_df.copy()

columns_to_drop = []

print("No additional columns dropped because unnecessary columns were not loaded.")
print("Shape before encoding:", model_df.shape)

model_df.head()

No additional columns dropped because unnecessary columns were not loaded.
Shape before encoding: (208213, 21)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balanceDiffOrig,balanceDiffDest,errorBalanceOrig,errorBalanceDest,isZeroBalanceAfterTransaction,isHighAmount,hourOfDay,isHighRiskType,logAmount,logBalanceDiffOrig,logBalanceDiffDest,logErrorBalanceOrig,logErrorBalanceDest
0,349,PAYMENT,65487.9883,141478.0000,75990.0078,0.0000,0.0000,0,65487.9922,0.0000,0.0078,65487.9883,0,0,13,0,11.0896,11.0896,0.0000,0.0078,11.0896
1,257,CASH_OUT,299940.0312,0.0000,0.0000,502460.1250,802400.1250,0,0.0000,299940.0000,-299940.0312,0.0000,1,0,17,1,12.6113,0.0000,12.6113,12.6113,0.0000
2,281,PAYMENT,16878.8594,17579.6504,700.7900,0.0000,0.0000,0,16878.8613,0.0000,0.0010,16878.8594,0,0,17,0,9.7339,9.7339,0.0000,0.0010,9.7339
3,42,CASH_OUT,132800.6250,0.0000,0.0000,638717.3125,771517.9375,0,0.0000,132800.6250,-132800.6250,0.0000,1,0,18,1,11.7966,0.0000,11.7966,11.7966,0.0000
4,15,CASH_OUT,238788.1562,0.0000,0.0000,672467.0625,911255.2500,0,0.0000,238788.1875,-238788.1562,0.0000,1,0,15,1,12.3833,0.0000,12.3833,12.3833,0.0000


In [40]:
# ---------------------------------------------------------
# Step 19: One-Hot Encode Transaction Type
# ---------------------------------------------------------
# The transaction type column is categorical.
# We convert it into numerical columns using one-hot encoding.

model_df_encoded = pd.get_dummies(
    model_df,
    columns=["type"],
    drop_first=False,
    dtype="int8"
)

print("Shape after one-hot encoding:", model_df_encoded.shape)

model_df_encoded.head()

Shape after one-hot encoding: (208213, 25)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balanceDiffOrig,balanceDiffDest,errorBalanceOrig,errorBalanceDest,isZeroBalanceAfterTransaction,isHighAmount,hourOfDay,isHighRiskType,logAmount,logBalanceDiffOrig,logBalanceDiffDest,logErrorBalanceOrig,logErrorBalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,349,65487.9883,141478.0000,75990.0078,0.0000,0.0000,0,65487.9922,0.0000,0.0078,65487.9883,0,0,13,0,11.0896,11.0896,0.0000,0.0078,11.0896,0,0,0,1,0
1,257,299940.0312,0.0000,0.0000,502460.1250,802400.1250,0,0.0000,299940.0000,-299940.0312,0.0000,1,0,17,1,12.6113,0.0000,12.6113,12.6113,0.0000,0,1,0,0,0
2,281,16878.8594,17579.6504,700.7900,0.0000,0.0000,0,16878.8613,0.0000,0.0010,16878.8594,0,0,17,0,9.7339,9.7339,0.0000,0.0010,9.7339,0,0,0,1,0
3,42,132800.6250,0.0000,0.0000,638717.3125,771517.9375,0,0.0000,132800.6250,-132800.6250,0.0000,1,0,18,1,11.7966,0.0000,11.7966,11.7966,0.0000,0,1,0,0,0
4,15,238788.1562,0.0000,0.0000,672467.0625,911255.2500,0,0.0000,238788.1875,-238788.1562,0.0000,1,0,15,1,12.3833,0.0000,12.3833,12.3833,0.0000,0,1,0,0,0


In [41]:
# ---------------------------------------------------------
# Step 21: Check Final Dataset Data Types
# ---------------------------------------------------------
# At this point, all columns should be numerical.

model_df_encoded.dtypes

,0
step,int16
amount,float32
oldbalanceOrg,float32
newbalanceOrig,float32
oldbalanceDest,float32
newbalanceDest,float32
isFraud,int8
balanceDiffOrig,float32
balanceDiffDest,float32
errorBalanceOrig,float32


In [42]:
# ---------------------------------------------------------
# Step 22: Separate Features and Target
# ---------------------------------------------------------
# X contains input features.
# y contains target labels.
#
# Target:
# isFraud
# 0 = Normal Transaction
# 1 = Fraud Transaction

X = model_df_encoded.drop(columns=["isFraud"])
y = model_df_encoded["isFraud"]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Feature matrix shape: (208213, 24)
Target vector shape: (208213,)

Target distribution:
isFraud
0    200000
1      8213
Name: count, dtype: int64


In [43]:
# ---------------------------------------------------------
# Step 23: Check Final Feature List
# ---------------------------------------------------------
# These are the final features that will be used for model training.

final_features = X.columns.tolist()

print("Total final features:", len(final_features))

for feature in final_features:
    print("-", feature)

Total final features: 24
- step
- amount
- oldbalanceOrg
- newbalanceOrig
- oldbalanceDest
- newbalanceDest
- balanceDiffOrig
- balanceDiffDest
- errorBalanceOrig
- errorBalanceDest
- isZeroBalanceAfterTransaction
- isHighAmount
- hourOfDay
- isHighRiskType
- logAmount
- logBalanceDiffOrig
- logBalanceDiffDest
- logErrorBalanceOrig
- logErrorBalanceDest
- type_CASH_IN
- type_CASH_OUT
- type_DEBIT
- type_PAYMENT
- type_TRANSFER


In [44]:
# ---------------------------------------------------------
# Step 24: Check Missing Values in Final Dataset
# ---------------------------------------------------------
# We confirm that no missing values exist in the final
# model-ready dataset.

missing_final = model_df_encoded.isnull().sum().sum()

print("Total missing values in final dataset:", missing_final)

if missing_final == 0:
    print("Final dataset is clean.")
else:
    print("Missing values found. Further cleaning required.")

Total missing values in final dataset: 0
Final dataset is clean.


In [45]:
# ---------------------------------------------------------
# Step 25: Check Infinite Values
# ---------------------------------------------------------
# Some mathematical transformations may create infinite values.
# We check and handle them if present.

numeric_columns = model_df_encoded.select_dtypes(include=[np.number]).columns

infinite_count = np.isinf(model_df_encoded[numeric_columns]).sum().sum()

print("Total infinite values:", infinite_count)

if infinite_count > 0:
    model_df_encoded.replace([np.inf, -np.inf], np.nan, inplace=True)
    model_df_encoded.fillna(0, inplace=True)
    print("Infinite values handled.")
else:
    print("No infinite values found.")

Total infinite values: 0
No infinite values found.


In [46]:
# ---------------------------------------------------------
# Step 26: Finalize Processed Dataset
# ---------------------------------------------------------
# The dataset is already optimized using smaller data types.
# We create model_df_optimized for consistent naming in later cells.

model_df_optimized = model_df_encoded.copy()

print("Final processed dataset prepared.")
print("Shape:", model_df_optimized.shape)

Final processed dataset prepared.
Shape: (208213, 25)


In [47]:
# ---------------------------------------------------------
# Step 26: Optimize Data Types
# ---------------------------------------------------------
# The dataset is large, so we reduce memory usage by converting
# integer and float columns to smaller data types where possible.

def optimize_dataframe(data):
    optimized_data = data.copy()

    for col in optimized_data.columns:
        col_type = optimized_data[col].dtype

        if col_type == "int64":
            optimized_data[col] = pd.to_numeric(optimized_data[col], downcast="integer")

        elif col_type == "float64":
            optimized_data[col] = pd.to_numeric(optimized_data[col], downcast="float")

    return optimized_data

memory_before = model_df_encoded.memory_usage(deep=True).sum() / (1024 ** 2)

model_df_optimized = optimize_dataframe(model_df_encoded)

memory_after = model_df_optimized.memory_usage(deep=True).sum() / (1024 ** 2)

print("Memory before optimization: {:.2f} MB".format(memory_before))
print("Memory after optimization: {:.2f} MB".format(memory_after))
print("Memory reduced by: {:.2f} MB".format(memory_before - memory_after))

Memory before optimization: 17.87 MB
Memory after optimization: 13.70 MB
Memory reduced by: 4.17 MB


In [48]:
# ---------------------------------------------------------
# Step 27: Final Processed Dataset Preview
# ---------------------------------------------------------
# This is the final machine-learning-ready dataset.

print("Final processed dataset shape:", model_df_optimized.shape)

model_df_optimized.head()

Final processed dataset shape: (208213, 25)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balanceDiffOrig,balanceDiffDest,errorBalanceOrig,errorBalanceDest,isZeroBalanceAfterTransaction,isHighAmount,hourOfDay,isHighRiskType,logAmount,logBalanceDiffOrig,logBalanceDiffDest,logErrorBalanceOrig,logErrorBalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,349,65487.9883,141478.0000,75990.0078,0.0000,0.0000,0,65487.9922,0.0000,0.0078,65487.9883,0,0,13,0,11.0896,11.0896,0.0000,0.0078,11.0896,0,0,0,1,0
1,257,299940.0312,0.0000,0.0000,502460.1250,802400.1250,0,0.0000,299940.0000,-299940.0312,0.0000,1,0,17,1,12.6113,0.0000,12.6113,12.6113,0.0000,0,1,0,0,0
2,281,16878.8594,17579.6504,700.7900,0.0000,0.0000,0,16878.8613,0.0000,0.0010,16878.8594,0,0,17,0,9.7339,9.7339,0.0000,0.0010,9.7339,0,0,0,1,0
3,42,132800.6250,0.0000,0.0000,638717.3125,771517.9375,0,0.0000,132800.6250,-132800.6250,0.0000,1,0,18,1,11.7966,0.0000,11.7966,11.7966,0.0000,0,1,0,0,0
4,15,238788.1562,0.0000,0.0000,672467.0625,911255.2500,0,0.0000,238788.1875,-238788.1562,0.0000,1,0,15,1,12.3833,0.0000,12.3833,12.3833,0.0000,0,1,0,0,0


In [49]:
# ---------------------------------------------------------
# Step 28: Save Processed Dataset as Parquet
# ---------------------------------------------------------
# Parquet is faster and more memory-efficient than CSV.
# This helps prevent RAM crashes in Google Colab.

PROCESSED_DATA_PATH = os.path.join(
    PROCESSED_DATA_DIR,
    "processed_fraud_data.parquet"
)

model_df_optimized.to_parquet(PROCESSED_DATA_PATH, index=False)

print("Processed dataset saved successfully in Parquet format.")
print("Saved path:", PROCESSED_DATA_PATH)
print("Final shape:", model_df_optimized.shape)

Processed dataset saved successfully in Parquet format.
Saved path: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/data/processed/processed_fraud_data.parquet
Final shape: (208213, 25)


In [50]:
# ---------------------------------------------------------
# Step 28.1: Save Small CSV Preview
# ---------------------------------------------------------
# We save only 1000 rows as CSV for quick checking.
# The full processed dataset is saved as Parquet.

PREVIEW_CSV_PATH = os.path.join(
    PROCESSED_DATA_DIR,
    "processed_fraud_data_preview.csv"
)

model_df_optimized.head(1000).to_csv(PREVIEW_CSV_PATH, index=False)

print("Small CSV preview saved successfully.")
print("Saved path:", PREVIEW_CSV_PATH)

Small CSV preview saved successfully.
Saved path: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/data/processed/processed_fraud_data_preview.csv


In [51]:
# ---------------------------------------------------------
# Step 29: Save Feature Metadata
# ---------------------------------------------------------
# This metadata will be used in Notebook 4 and Streamlit app.

X = model_df_optimized.drop(columns=["isFraud"])
final_features = X.columns.tolist()

FEATURE_LIST_PATH = os.path.join(PROCESSED_DATA_DIR, "feature_list.json")

feature_metadata = {
    "features": final_features,
    "target": "isFraud",
    "high_amount_threshold": float(HIGH_AMOUNT_THRESHOLD),
    "normal_sample_size": NORMAL_SAMPLE_SIZE,
    "high_risk_types": high_risk_types,
    "processed_file": "processed_fraud_data.parquet"
}

with open(FEATURE_LIST_PATH, "w") as file:
    json.dump(feature_metadata, file, indent=4)

print("Feature metadata saved successfully.")
print("Total features:", len(final_features))
print("Saved path:", FEATURE_LIST_PATH)

Feature metadata saved successfully.
Total features: 24
Saved path: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/data/processed/feature_list.json


In [52]:
# ---------------------------------------------------------
# Step 30: Verify Saved Files
# ---------------------------------------------------------
# We check whether processed dataset and feature metadata were
# saved correctly.

print("Files inside data/processed:")

for file in os.listdir(PROCESSED_DATA_DIR):
    print("-", file)

Files inside data/processed:
- eda_sample.csv
- processed_fraud_data.csv
- feature_list.json
- processed_fraud_data.parquet
- processed_fraud_data_preview.csv


In [53]:
# ---------------------------------------------------------
# Step 31: Reload Processed Dataset for Verification
# ---------------------------------------------------------
# We reload the Parquet file to confirm that the saved file
# is readable and correct.

processed_check = pd.read_parquet(PROCESSED_DATA_PATH)

print("Parquet file loaded successfully.")
print("Reloaded processed dataset shape:", processed_check.shape)

processed_check.head()

Parquet file loaded successfully.
Reloaded processed dataset shape: (208213, 25)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balanceDiffOrig,balanceDiffDest,errorBalanceOrig,errorBalanceDest,isZeroBalanceAfterTransaction,isHighAmount,hourOfDay,isHighRiskType,logAmount,logBalanceDiffOrig,logBalanceDiffDest,logErrorBalanceOrig,logErrorBalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,349,65487.9883,141478.0000,75990.0078,0.0000,0.0000,0,65487.9922,0.0000,0.0078,65487.9883,0,0,13,0,11.0896,11.0896,0.0000,0.0078,11.0896,0,0,0,1,0
1,257,299940.0312,0.0000,0.0000,502460.1250,802400.1250,0,0.0000,299940.0000,-299940.0312,0.0000,1,0,17,1,12.6113,0.0000,12.6113,12.6113,0.0000,0,1,0,0,0
2,281,16878.8594,17579.6504,700.7900,0.0000,0.0000,0,16878.8613,0.0000,0.0010,16878.8594,0,0,17,0,9.7339,9.7339,0.0000,0.0010,9.7339,0,0,0,1,0
3,42,132800.6250,0.0000,0.0000,638717.3125,771517.9375,0,0.0000,132800.6250,-132800.6250,0.0000,1,0,18,1,11.7966,0.0000,11.7966,11.7966,0.0000,0,1,0,0,0
4,15,238788.1562,0.0000,0.0000,672467.0625,911255.2500,0,0.0000,238788.1875,-238788.1562,0.0000,1,0,15,1,12.3833,0.0000,12.3833,12.3833,0.0000,0,1,0,0,0


In [54]:
# ---------------------------------------------------------
# Step 32: Reload Feature Metadata
# ---------------------------------------------------------
# This confirms that feature metadata was saved correctly.

with open(FEATURE_LIST_PATH, "r") as file:
    loaded_metadata = json.load(file)

print("Target column:", loaded_metadata["target"])
print("Number of features:", len(loaded_metadata["features"]))
print("High amount threshold:", loaded_metadata["high_amount_threshold"])

print("\nFirst 10 features:")
loaded_metadata["features"][:10]

Target column: isFraud
Number of features: 24
High amount threshold: 652597.4249999999

First 10 features:


['step',
 'amount',
 'oldbalanceOrg',
 'newbalanceOrig',
 'oldbalanceDest',
 'newbalanceDest',
 'balanceDiffOrig',
 'balanceDiffDest',
 'errorBalanceOrig',
 'errorBalanceDest']

In [55]:
# ---------------------------------------------------------
# Step 33: Save Feature Engineering Report
# ---------------------------------------------------------
# We save a report summarizing all feature engineering work.

feature_report_path = os.path.join(REPORTS_DIR, "03_feature_engineering_report.txt")

with open(feature_report_path, "w") as file:
    file.write("UPI Shield AI — Feature Engineering Report\n")
    file.write("=" * 55 + "\n\n")

    file.write("Data Strategy:\n")
    file.write("Loaded only required columns from the raw dataset to reduce RAM usage.\n")
    file.write("Kept all fraud records and sampled normal records for modeling.\n\n")

    file.write("Normal Sample Size:\n")
    file.write(f"{NORMAL_SAMPLE_SIZE}\n\n")

    file.write("Processed Dataset Shape:\n")
    file.write(f"{model_df_optimized.shape}\n\n")

    file.write("Created Features:\n")
    for feature in engineered_features:
        file.write(f"- {feature}\n")

    file.write("\nFinal Model Features:\n")
    for feature in final_features:
        file.write(f"- {feature}\n")

    file.write("\nTarget Column:\n")
    file.write("isFraud\n\n")

    file.write("High Amount Threshold:\n")
    file.write(str(HIGH_AMOUNT_THRESHOLD))

    file.write("\n\nSaved Files:\n")
    file.write("- processed_fraud_data.parquet\n")
    file.write("- processed_fraud_data_preview.csv\n")
    file.write("- feature_list.json\n")

    file.write("\n\nImportant Note:\n")
    file.write("SMOTE was not applied in this notebook. SMOTE will be applied only on training data after train-test split in Notebook 4 to avoid data leakage.\n")

print("Feature engineering report saved successfully.")
print("Report path:", feature_report_path)

Feature engineering report saved successfully.
Report path: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/reports/03_feature_engineering_report.txt


In [56]:
# ---------------------------------------------------------
# Step 34: Final Notebook Summary
# ---------------------------------------------------------
# This confirms that Notebook 3 has been completed successfully.

print("Notebook 3: Feature Engineering completed successfully.")

print("\nKey Outputs:")
print("- Balance difference features created")
print("- Balance error features created")
print("- Zero balance indicator created")
print("- High amount indicator created")
print("- Hour of day feature created")
print("- High-risk transaction type feature created")
print("- Categorical transaction type encoded")
print("- ID and leakage columns removed")
print("- Final processed dataset saved")
print("- Feature metadata saved")
print("- Feature engineering report saved")

print("\nImportant:")
print("SMOTE was not applied here.")
print("SMOTE will be applied only on training data in Notebook 4.")

print("\nNext Notebook:")
print("04_model_training.ipynb")

Notebook 3: Feature Engineering completed successfully.

Key Outputs:
- Balance difference features created
- Balance error features created
- Zero balance indicator created
- High amount indicator created
- Hour of day feature created
- High-risk transaction type feature created
- Categorical transaction type encoded
- ID and leakage columns removed
- Final processed dataset saved
- Feature metadata saved
- Feature engineering report saved

Important:
SMOTE was not applied here.
SMOTE will be applied only on training data in Notebook 4.

Next Notebook:
04_model_training.ipynb


# Notebook 3 Summary

In this notebook, we completed feature engineering for the UPI Shield AI project.

## Features Created

1. `balanceDiffOrig`
2. `balanceDiffDest`
3. `errorBalanceOrig`
4. `errorBalanceDest`
5. `isZeroBalanceAfterTransaction`
6. `isHighAmount`
7. `hourOfDay`
8. `isHighRiskType`
9. `logAmount`
10. `logBalanceDiffOrig`
11. `logBalanceDiffDest`
12. `logErrorBalanceOrig`
13. `logErrorBalanceDest`

## Columns Removed

1. `nameOrig`
2. `nameDest`
3. `isFlaggedFraud`

## Why SMOTE Was Not Applied Here

SMOTE creates synthetic minority class samples.

If SMOTE is applied before train-test split, synthetic samples may leak information into the test set. This causes data leakage and gives unrealistic model performance.

Therefore, SMOTE will be applied only after train-test split and only on the training data in Notebook 4.

## Output Files Created

1. `processed_fraud_data.csv`
2. `feature_list.json`
3. `03_feature_engineering_report.txt`

## Next Step

The next notebook will be:

`04_model_training.ipynb`